# Training with both RNA-Seq and Microarray data
We have seen that there is significant performance drop when training with both. Here we will go through all the filtering steps and evaluate how much data we have with different filtering thresholds. 

In [6]:
import scanpy as sc
import sys
sys.path.append("../../")
from src.training import helpers as tr_h
adata = sc.read("/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-09-10-01/data.h5ad")
mask_genes = tr_h.get_top_k_most_present_genes(
    adata, k=3501
)


In [7]:
adata = adata[:, mask_genes]

In [53]:
import numpy as np

nan_thr= 0.9

non_nan_mask = ~np.isnan(adata.X)  & ~(adata.X==0) 
non_nan_mask_pct = np.sum(non_nan_mask, axis=1) / adata.X.shape[1]

# mask samples that have less than 30% non-NaN values
mask_samples_nan = non_nan_mask_pct >= nan_thr 

print(f"{nan_thr} Keeping {np.sum(mask_samples_nan)} samples out of {adata.X.shape[0]} ({np.sum(mask_samples_nan)/adata.X.shape[0]*100:.2f}%)")


zero_thr = 0.5
non_zero_mask = ~(adata.X==0) 
non_zero_mask_pct = np.sum(non_zero_mask, axis=1) / adata.X.shape[1]

# mask samples that have less than 30% non-NaN values
mask_samples_zero = non_zero_mask_pct >= zero_thr 

print(f"{zero_thr} Keeping {np.sum(mask_samples_zero)} samples out of {adata.X.shape[0]} ({np.sum(mask_samples_zero)/adata.X.shape[0]*100:.2f}%)")

mask_samples_comb = mask_samples_nan & mask_samples_zero
print(f"Combined: Keeping {np.sum(mask_samples_comb)} samples out of {adata.X.shape[0]} ({np.sum(mask_samples_comb)/adata.X.shape[0]*100:.2f}%)")

0.9 Keeping 100610 samples out of 111082 (90.57%)
0.5 Keeping 109791 samples out of 111082 (98.84%)
Combined: Keeping 100610 samples out of 111082 (90.57%)


In [9]:
for pct_thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    non_zero_non_nan_mask = ~(adata.X==0) 
    non_zero_non_nan_mask_pct = np.sum(non_zero_non_nan_mask, axis=1) / adata.X.shape[1]

    # mask samples that have less than 30% non-NaN values
    mask_samples = non_zero_non_nan_mask_pct >= pct_thr 

    print(f"{pct_thr} Keeping {np.sum(mask_samples)} samples out of {adata.X.shape[0]} ({np.sum(mask_samples)/adata.X.shape[0]*100:.2f}%)")

0.3 Keeping 110630 samples out of 111082 (99.59%)
0.4 Keeping 110355 samples out of 111082 (99.35%)
0.5 Keeping 109791 samples out of 111082 (98.84%)
0.6 Keeping 109061 samples out of 111082 (98.18%)
0.7 Keeping 108535 samples out of 111082 (97.71%)
0.8 Keeping 108078 samples out of 111082 (97.30%)
0.9 Keeping 106747 samples out of 111082 (96.10%)


In [ ]:

# apply the mask to the AnnData object
adata[mask_samples, :].shape()


In [18]:
filtered_adata  = tr_h.clean_adata_qc(adata,n_samples=1, n_dt=4)


Nº of datasets with +1 control samples: 2111
Nº of datasets with +1 disease samples: 2111
Nº of datasets with +1 samples (control and disease): 2111
adata shape after filtering datasets with +1 samples: (111082, 3501)
Nº of passed diseases 142/ 292
Nº of passed dsaids 3395/ 3927


In [15]:
filtered_adata.obs["doid_id"].nunique()

196

In [45]:
adata_f.obs.query('doid_id == "DOID:1578"')["dataset_id"].unique()

['GSE166059', 'GSE40839', 'GSE81292']
Categories (1958, object): ['E-MEXP-3097', 'E-MTAB-567', 'E-MTAB-1030', 'E-MTAB-1132', ..., 'GSE224056', 'GSE225904', 'GSE226869', 'GSE227329']

In [50]:
adata_f.obs.query("dataset_id == 'GSE40839'")["doid_id"].unique()

['Control', 'DOID:1578']
Categories (196, object): ['Control', 'DOID:235', 'DOID:289', 'DOID:299', ..., 'DOID:0060224', 'DOID:0060901', 'DOID:0080199', 'DOID:0081087']

In [51]:
import importlib
importlib.reload(tr_h)

<module 'src.training.helpers' from '/aloy/home/ddalton/projects/scGPT_playground/notebooks/exp/../../src/training/helpers.py'>

In [ ]:
adata_q = adata[mask_samples, :]


adata_f = tr_h.clean_adata_qc(adata_q, n_samples=1, n_dt=3)
print(f"Nº of diseases after filtering: {adata_f.obs['doid_id'].nunique()}")


# split
df_obs = adata_f.obs
test_obs = tr_h.split_stratified(
    df=df_obs,
    y_label="doid_id",  # should ALWAYS be on DOID! - OR  celltype be DOID! 
    group_label="dataset_id",
    split_size=10,
    seed=42,
)
tr_h.report_split(test_obs, disease_label="doid_id")





valid_obs = tr_h.split_stratified(
    df=test_obs[test_obs["test_split_1"]==0],
    y_label="doid_id",  # should ALWAYS be on DOID! - OR  celltype be DOID! 
    group_label="dataset_id",
    split_size=10,
    seed=42,
    new_label="valid_split_1"
)
tr_h.report_split(valid_obs, disease_label="doid_id", split_label="valid_split_1")



Nº of datasets with +1 control samples: 1911
Nº of datasets with +1 disease samples: 1912
Nº of datasets with +1 samples (control and disease): 1905
adata shape after filtering datasets with +1 samples: (100587, 3501)
Nº of passed diseases 112/ 291
Nº of passed dsaids 2903/ 3566
Nº of diseases after filtering: 113
df shape: (83952, 16)
df diseases: (46834, 16)
All Labels: 113 ['Control', 'DOID:0050156', 'DOID:0050700', 'DOID:0080199', 'DOID:0081087', 'DOID:10223', 'DOID:1024', 'DOID:10286', 'DOID:1040', 'DOID:10591', 'DOID:10608', 'DOID:10652', 'DOID:10763', 'DOID:10923', 'DOID:10941', 'DOID:11335', 'DOID:11555', 'DOID:11612', 'DOID:11714', 'DOID:11722', 'DOID:11723', 'DOID:11725', 'DOID:11727', 'DOID:11934', 'DOID:12177', 'DOID:12377', 'DOID:12704', 'DOID:12858', 'DOID:12894', 'DOID:12930', 'DOID:1312', 'DOID:1319', 'DOID:13223', 'DOID:13241', 'DOID:13378', 'DOID:13922', 'DOID:14203', 'DOID:14221', 'DOID:14250', 'DOID:14330', 'DOID:1485', 'DOID:1520', 'DOID:1749', 'DOID:1883', 'DOID:1

In [39]:
adata_f.obs.groupby("doid_id")["dataset"].nunique().sort_values()

/tmp/ipykernel_153703/2624498825.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata_f.obs.groupby("doid_id")["dataset"].nunique().sort_values()


doid_id
DOID:0060901       3
DOID:676           3
DOID:0060224       3
DOID:0050427       3
DOID:1107          3
                ... 
DOID:10652        59
DOID:9074         63
DOID:2841         64
DOID:14330        66
Control         1958
Name: dataset, Length: 196, dtype: int64

In [42]:
df = adata_f.obs
y_label = "doid_id" 
group_label = "dataset_id"

# (tiny guard) each label must span ≥2 datasets to appear in both splits
_df_diseases = df[df[y_label] != "Control"]
print(f"df diseases: {_df_diseases.shape}")

ds_per_label = _df_diseases.groupby(y_label, observed=True)[group_label].nunique()
if (ds_per_label < 2).any():
    print(ds_per_label[ds_per_label < 2])
    raise ValueError("Some labels occur in <2 datasets; cannot place them in both splits.")


df diseases: (57716, 16)


In [28]:
test_obs

,ids,dataset,dataset_id,batch,batch_id,dsaid,tissue,n_genes,disease,celltype,disease_study,library,doid_study,doid_id,do_id,doid_disease,test_split_1
9,DSA00006.GSM3596906.Control,GSE126342,GSE126342,1512,1512,DSA00006,Skeletal muscle,19402,Control,Control,Myotonic Dystrophy Type 1,RNA-Seq,D,Control,Control,Control,0
10,DSA00006.GSM3596907.Control,GSE126342,GSE126342,1512,1512,DSA00006,Skeletal muscle,19402,Control,Control,Myotonic Dystrophy Type 1,RNA-Seq,D,Control,Control,Control,0
11,DSA00006.GSM3596908.Control,GSE126342,GSE126342,1512,1512,DSA00006,Skeletal muscle,19402,Control,Control,Myotonic Dystrophy Type 1,RNA-Seq,D,Control,Control,Control,0
12,DSA00006.GSM3596909.Control,GSE126342,GSE126342,1512,1512,DSA00006,Skeletal muscle,19402,Control,Control,Myotonic Dystrophy Type 1,RNA-Seq,D,Control,Control,Control,0
13,DSA00006.GSM3596910.Control,GSE126342,GSE126342,1512,1512,DSA00006,Skeletal muscle,19402,Control,Control,Myotonic Dystrophy Type 1,RNA-Seq,D,Control,Control,Control,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111077,DSA10302.GSM139471.Case,GSE6008,GSE6008,1354,1354,DSA10302,Ovary,12004,Ovarian Tumor,Ovarian Tumor,Ovarian Tumor,Microarray,D,DOID:2394,DOID:2394,ovarian cancer,0
111078,DSA10302.GSM139472.Case,GSE6008,GSE6008,1354,1354,DSA10302,Ovary,12004,Ovarian Tumor,Ovarian Tumor,Ovarian Tumor,Microarray,D,DOID:2394,DOID:2394,ovarian cancer,0
111079,DSA10302.GSM139473.Case,GSE6008,GSE6008,1354,1354,DSA10302,Ovary,12004,Ovarian Tumor,Ovarian Tumor,Ovarian Tumor,Microarray,D,DOID:2394,DOID:2394,ovarian cancer,0
111080,DSA10302.GSM139474.Case,GSE6008,GSE6008,1354,1354,DSA10302,Ovary,12004,Ovarian Tumor,Ovarian Tumor,Ovarian Tumor,Microarray,D,DOID:2394,DOID:2394,ovarian cancer,0


In [20]:
adata_f.obs["doid_id"].nunique()

143